# 09 — Advanced: Hooks & Lifecycle

**Stage 9 of the workshop (Production, extended).** Observing and controlling the agent loop from the outside via lifecycle hooks.

## Problem

Getting from "works on my laptop" to something a team can rely on. Hooks are the mechanism for bolting on timing, logging, and guardrail-style checks without touching the Agent's own code — a production concern, not a core-loop concern.

## Concept

Hooks are callbacks fired at specific points in the agent loop: before/after the whole invocation, before/after each model call, before/after each tool call, and whenever a message is added to the conversation. They don't replace the loop — they observe (and can influence) it from outside.

The agent logic doesn't change when you add hooks — same architectural pattern as the model-provider swap: attach behavior externally, core code stays untouched.

## Architecture

```
agent("What is 47 plus 55? Use the add tool.")
     │
     ├─ BeforeModelCallEvent ──▶ start_model_timer()
     ├─ [model call]
     ├─ AfterModelCallEvent ──▶ report_model_duration()
     │
     ├─ BeforeToolCallEvent ──▶ log_tool_start()      "tool call starting: add"
     ├─ [tool call: add(47, 55)]
     ├─ AfterToolCallEvent ──▶ log_tool_end()          "tool call finished: add"
     │
     ├─ [model call to produce final answer]
     ▼
  final answer
```

All four hooks are pure side-effect functions registered via `agent.add_hook(...)` — none of them alter the tool/model logic itself.

## Step 1 — Model setup and hook imports

In [1]:
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from model_provider import get_model
from strands import Agent, tool
from strands.hooks import (
    AfterModelCallEvent,
    AfterToolCallEvent,
    BeforeModelCallEvent,
    BeforeToolCallEvent,
)

model = get_model()

_timers = {}


## Step 2 — Define the hook callbacks

Each hook receives a typed event object and can read from / react to it — here purely for timing and logging.

In [2]:
def start_model_timer(event: BeforeModelCallEvent) -> None:
    _timers["model"] = time.perf_counter()


def report_model_duration(event: AfterModelCallEvent) -> None:
    start = _timers.get("model")
    if start is not None:
        print(f"[hook] model call took {time.perf_counter() - start:.2f}s")


def log_tool_start(event: BeforeToolCallEvent) -> None:
    print(f"[hook] tool call starting: {event.tool_use['name']}")


def log_tool_end(event: AfterToolCallEvent) -> None:
    print(f"[hook] tool call finished: {event.tool_use['name']}")


## Step 3 — Define the tool and agent, register hooks

In [3]:
@tool
def add(x: int, y: int) -> int:
    """Add two numbers."""
    return x + y


agent = Agent(model=model, tools=[add])
agent.add_hook(start_model_timer)
agent.add_hook(report_model_duration)
agent.add_hook(log_tool_start)
agent.add_hook(log_tool_end)


## Step 4 — Run it

Watch for the `[hook]` lines interleaved with the agent's own output — they fire from outside the loop, at each lifecycle point.

In [4]:
result = agent("What is 47 plus 55? Use the add tool.")
print(result)



Tool #1: add
[hook] model call took 1.04s
[hook] tool call starting: add
[hook] tool call finished: add


The result of 47 plus 55 is 10

2.[hook] model call took 0.65s
The result of 47 plus 55 is 102.

